# 07 - final evaluation on the test glaciers (Columbia + Mapple)

**GPU runtime, high RAM. Run once.** The test set has not been touched by any
notebook before this one. Every model gets scored on the same 122 test images,
with no choices made after seeing test numbers:

| model | how |
|---|---|
| frozen encoders (5; the satellite DINOv3 was added on 2026-09-25, after the first test run, see README, Protocol) | features from train + test in one pass; 3-class probe (patch level, as in 03) and 4-class probe -> zone maps -> official front route -> MDE |
| U-Net 3-class, 48k | saved weights from `05`, no retraining |
| U-Net 4-class, 48k | saved weights from `06`, no retraining |

Metrics reported:
- **MDE (m)** per test glacier and per sensor -- comparable with the paper
- **paper-style zone IoU**: per-image IoU averaged over images, 4 classes
  including NA (the paper's definition), plus our pooled pixel IoU
- **3-class patch mIoU** for continuity with the SI tables

**Published, test set, from the papers (sources in the README):**
CaFFe baseline (U-Net + ASPP, mean of 5 runs): zone IoU ocean 0.82, glacier
0.74, rock 0.68, NA 0.93, mean 0.79; MDE on all 122 images 753 +/- 76 m,
Mapple 150 +/- 24 m, Columbia 840 +/- 84 m. AMD-HookNet: MDE 438 m.
HookFormer: MDE 353 +/- 16 m. AMD-HookNet++: MDE 367 m, IoU 78.2.

In [7]:
# --- cell: mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# <<<SARTRANSFER-SYNC-BEGIN>>>
# Empty sync cell: published notebooks carry no packed code. On your own machine,
# in the repository folder, run   python sync.py   and open the notebook again;
# this cell then holds the package source and unpacks it on the runtime.
raise RuntimeError("The sync cell is empty. On your own machine, in the repository folder, "
                   "run:  python sync.py  and reopen this notebook.")
# <<<SARTRANSFER-SYNC-END>>>

In [ ]:
# --- cell: secrets, GPU, data ---
import os, shutil
from pathlib import Path
import torch
from sartransfer.env import load_env, require
from sartransfer.data import inventory as inv

load_env()
require("HF_TOKEN", "gated model weights")
assert torch.cuda.is_available(), "switch to a GPU runtime"
print("gpu:", torch.cuda.get_device_properties(0).name)
DRIVE = Path("/content/drive/MyDrive")
RAW, RES = DRIVE / "sar-transfer" / "raw", DRIVE / "sar-transfer" / "results"
CAFFE = Path("/content/caffe")
inv.unzip_caffe(inv.fetch_caffe_zip(RAW), CAFFE)
DATA = inv.find_data_root(CAFFE)
df = inv.scan(DATA, subdirs=("sar_images",))
n_test = df["path"].str.contains("/sar_images/test/").sum()
print(len(df), "SAR images,", n_test, "in the test folders (expect 122)")
assert n_test == 122

In [10]:
# --- cell: FROZEN ENCODERS on the test glaciers (one loop: 25 min for the first four encoders, 6.5 min for the satellite DINOv3) ---
import time
from sartransfer.runs import run_final_encoder

RUNS = [("cradio-v4-h", "caffe"), ("dinov3-l-photo", "caffe"), ("dinov3-l-sat", "caffe"),
        ("saratrx-v1", "unit"), ("terramind-v1-base-s1", "caffe")]
t0, status = time.perf_counter(), {}
for encoder, norm in RUNS:
    try:
        done = run_final_encoder(encoder, norm, df, DATA, RES, token=os.environ["HF_TOKEN"])
        status[f"{encoder} | {norm}"] = "done" if done else "skipped (already saved)"
    except Exception as e:
        status[f"{encoder} | {norm}"] = f"FAILED: {type(e).__name__}: {str(e)[:300]}"
        print(status[f"{encoder} | {norm}"])
print(f"\n===== encoders finished in {(time.perf_counter() - t0) / 60:.0f} min =====")
for k, v in status.items():
    print(f"  {k:<32} {v}")

skip cradio-v4-h__caffe__tile512: test results already on Drive
skip dinov3-l-photo__caffe__tile512: test results already on Drive

===== TEST dinov3-l-sat__caffe__tile512 =====


Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

dinov3-l-sat: 303.1 M parameters loaded
restricted to splits ('train', 'test'): 560 of 681 images
  50/560  0.5 min elapsed, 0.61 s/image, ETA 5.2 min
  100/560  0.6 min elapsed, 0.34 s/image, ETA 2.6 min
  150/560  0.7 min elapsed, 0.29 s/image, ETA 2.0 min
  200/560  1.1 min elapsed, 0.34 s/image, ETA 2.1 min
  250/560  1.2 min elapsed, 0.30 s/image, ETA 1.5 min
  300/560  1.5 min elapsed, 0.30 s/image, ETA 1.3 min
  350/560  2.0 min elapsed, 0.35 s/image, ETA 1.2 min
  400/560  2.5 min elapsed, 0.38 s/image, ETA 1.0 min
  450/560  3.1 min elapsed, 0.41 s/image, ETA 0.7 min
  500/560  3.4 min elapsed, 0.41 s/image, ETA 0.4 min
  550/560  3.7 min elapsed, 0.40 s/image, ETA 0.1 min
  560/560  3.7 min elapsed, 0.40 s/image, ETA 0.0 min
in memory: train X (2148306, 1024) (4.4 GB fp16), 122 test images
TEST: MDE 2805 m (60 of 122 images without a front) | per glacier {'COL': 3563.0, 'Mapple': 526.0} | paper-style mIoU 0.595
  3-class patch mIoU on test: 0.5856 | 6.5 min
skip saratrx-v1__u

In [11]:
# --- cell: SAVED U-NETS on the test glaciers (no retraining, ~5 min) ---
from sartransfer.baseline import load_split_images
from sartransfer.runs import run_final_unet

te3 = load_split_images(df, DATA, "test", norm="caffe")
te4 = load_split_images(df, DATA, "test", norm="caffe", with_na=True)
assert len(te3) == 122 and len(te4) == 122
print("test images:", len(te3), "| glaciers:", sorted({im["glacier"] for im in te3}))

run_final_unet(RES / "unet__unet-scratch-48k__caffe__tile256.pt", 3, te3, DATA, RES,
               "unet-scratch-48k__caffe__tile256")
run_final_unet(RES / "unet__unet-scratch-48k__caffe__tile256__na4.pt", 4, te4, DATA, RES,
               "unet-scratch-48k__caffe__tile256__na4")
del te3, te4

test images: 122 | glaciers: ['COL', 'Mapple']
skip unet-scratch-48k__caffe__tile256: test results already on Drive
skip unet-scratch-48k__caffe__tile256__na4: test results already on Drive


In [12]:
# --- cell: FINAL TABLES ---
import glob
import pandas as pd
pd.set_option("display.width", 220, "display.precision", 3)

def load(pattern):
    fs = sorted(glob.glob(str(RES / pattern)))
    return pd.concat([pd.read_csv(f) for f in fs], ignore_index=True) if fs else pd.DataFrame()

mde = load("TEST_fronts_mde__*.csv")
paper = load("TEST_zones4_paper__*.csv")
z3 = load("TEST_probe_zones3__*.csv")

print("=== MDE in metres on the TEST glaciers (lower is better) ===")
t = mde[mde.grouping.isin(["all", "glacier"])].pivot_table(index="run", columns="group", values="MDE_m")
nf = mde[mde.grouping == "all"].set_index("run")["no_front"]
t["no_front_of_122"] = nf
print(t[["all", "COL", "Mapple", "no_front_of_122"]].sort_values("all").round(0).to_string())
print("published: CaFFe baseline Mapple 150 +/- 24, Columbia 840 +/- 84; AMD-HookNet 438; AMD-HookNet++ 367")

print("\n=== paper-style zone IoU on TEST: per-image IoU averaged over images, 4 classes incl. NA ===")
pt = paper[paper.grouping == "all"].set_index("run")[["IoU_na", "IoU_stone", "IoU_glacier", "IoU_ocean", "mIoU"]]
print(pt.sort_values("mIoU", ascending=False).round(3).to_string())
print("published CaFFe baseline: NA 0.93, rock 0.68, glacier 0.74, ocean 0.82, mean 0.79")

print("\n=== 3-class patch mIoU on TEST (same definition as the SI tables) ===")
p3 = z3[(z3.grouping == "glacier")].pivot_table(index="run", columns="group", values="mIoU")
print(p3[["all", "COL", "Mapple"]].sort_values("all", ascending=False).round(3).to_string())

print("\n=== MDE per sensor @ resolution on TEST (m) ===")
print(mde[mde.grouping == "sensor_res"].pivot_table(index="run", columns="group", values="MDE_m").round(0).to_string())

=== MDE in metres on the TEST glaciers (lower is better) ===
group                                     all     COL  Mapple  no_front_of_122
run                                                                           
unet-scratch-48k__caffe__tile256__na4   979.0  1080.0   474.0                3
cradio-v4-h__caffe__tile512            2089.0  2349.0   505.0               37
dinov3-l-photo__caffe__tile512         2199.0  2302.0   823.0               48
terramind-v1-base-s1__caffe__tile512   2414.0  3095.0   908.0               55
saratrx-v1__unit__tile512              2796.0  3313.0   921.0               70
dinov3-l-sat__caffe__tile512           2805.0  3563.0   526.0               60
published: CaFFe baseline Mapple 150 +/- 24, Columbia 840 +/- 84; AMD-HookNet 438; AMD-HookNet++ 367

=== paper-style zone IoU on TEST: per-image IoU averaged over images, 4 classes incl. NA ===
                                       IoU_na  IoU_stone  IoU_glacier  IoU_ocean   mIoU
run                     

## Reading it

- The published MDEs are from a **different pipeline** (U-Net + ASPP with front
  and zone models, 5-run means). Ours is a plain U-Net / linear probes, single
  run, official post-processing ported and verified. Compare cautiously.
- `no_front_of_122`: images where a model drew no front are excluded from MDE,
  as in the official code. A low MDE with many missing fronts is not better.
- Paper-style IoU includes NA (the best class for every model: 0.90 U-Net,
  0.55-0.77 probes), which lifts the mean; the SI tables in 04 excluded it.
  Do not compare the two directly.
- This notebook is run once. If a bug is found later, fix it, record the fix
  with its date, and re-run with `force=True` -- but never tune anything on these numbers.